In [215]:
import numpy as np
from functools import partial

In [229]:
img = np.arange(1, 26).reshape(5, 5)
img

array([[ 1,  2,  3,  4,  5],
       [ 6,  7,  8,  9, 10],
       [11, 12, 13, 14, 15],
       [16, 17, 18, 19, 20],
       [21, 22, 23, 24, 25]])

In [230]:
# Задание 1a. Линейный перебор
def linear(width, height):
    for h in range(height):
        for w in range(width):
            yield h, w 
            
            
for h, w in linear(5, 5):
    print(img[h][w])

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25


In [231]:
# Задание 1b. Перебор по спирали 
def spiral(width, height):
    y, x = height // 2, width // 2
    yield (y, x)
    
    step = 1
    while (0 <= x < height) and (0 <= y < width):
        for _ in range(step):
            x -= 1
            if (0 <= x < width and 0 <= y < height):
                yield y, x

        for _ in range(step):
            y -= 1
            if (0 <= x < width and 0 <= y < height):
                yield y, x

        step += 1

        for _ in range(step):
            x += 1
            if (0 <= x < width and 0 <= y < height):
                yield y, x

        for _ in range(step):
            y += 1
            if (0 <= x < width and 0 <= y < height):
                yield y, x
            
        step += 1

        for _ in range(step):
            x -= 1
            if (0 <= x < width and 0 <= y < height):
                yield y, x
            
        for _ in range(step):
            y -= 1
            if (0 <= x < width and 0 <= y < height):
                yield y, x
            
        step += 1
        
        for _ in range(step):
            x += 1
            if (0 <= x < width and 0 <= y < height):
                yield y, x
        
        for _ in range(step):
            y += 1
            if (0 < x < width and 0 < y < height):
                yield y, x
                
print(img)
for h, w in spiral(5, 5):
    print(img[h][w])
    


[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]
 [16 17 18 19 20]
 [21 22 23 24 25]]
13
12
7
8
9
14
19
18
17
16
11
6
1
2
3
4
5
10
15
20
25
24
23
22
21
16
11
6
1
2
3
4
5


In [232]:
# Задание 1c. Проход зиг-загом 
def zigzag(width, height):
    for s in range(width + height - 1):
        if s % 2 == 0:
            x = min(s, width - 1)
            y = s - x
            while x >= 0 and y < height:
                yield (x, y)
                x -= 1
                y += 1
        else:
            y = min(s, height - 1)
            x = s - y
            while y >= 0 and x < width:
                yield (x, y)
                x += 1
                y -= 1
                
print(img)
for h, w in zigzag(5, 5):
    print(img[h][w])

[[ 1  2  3  4  5]
 [ 6  7  8  9 10]
 [11 12 13 14 15]
 [16 17 18 19 20]
 [21 22 23 24 25]]
1
2
6
11
7
3
4
8
12
16
21
17
13
9
5
10
14
18
22
23
19
15
20
24
25


In [ ]:
# Задание 2. Конвертация изображения в оттенки серого
to_gray = lambda r, g, b: 0.299 * r + 0.587 * g + 0.114 * b

def rgb_to_gray(img):
    height, width, _ = img.shape
    out = np.empty((h, w))
    for r in range(h):
        for c in range(w):
            out[r, c] = to_gray(*img[r, c])  
    return out

In [ ]:
# Задание 3. Свёрточный оператор 

avg_kernel = np.array([[1/9]*3 for _ in range(3)])
gauss_kernel = np.array([[1, 2, 1],
                         [2, 4, 2],
                         [1, 2, 1]], dtype=float) / 16.0

def convolve(img, traversal, kernel):
    h, w = img.shape
    kh, kw = kernel.shape
    rh, rw = kh // 2, kw // 2
    out = img.copy() 

    for r, c in traversal(h, w):
        if r - rh < 0 or r + rh >= h or c - rw < 0 or c + rw >= w:
            continue
        window = img[r - rh:r + rh + 1, c - rw:c + rw + 1]
        out[r, c] = np.sum(window * kernel)
    return out

avg_filter   = partial(convolve, kernel=avg_kernel)
gauss_filter = partial(convolve, kernel=gauss_kernel)

In [ ]:
# Задание 4

def mse(a, b):
    return float(np.mean((a - b) ** 2))

if __name__ == "__main__":
    img_rgb = (np.random.rand(10, 10, 3) * 255).astype(float)
    img_gray = rgb_to_gray(img_rgb)
    traversals = {
        "linear": linear,
        "spiral": spiral,
        "zigzag": zigzag,
    }

    
    print("Усреднение:")
    base_avg = avg_filter(img_gray, linear) 
    for name, func in traversals.items():
        res = avg_filter(img_gray, func)
        print(f"MSE({name} vs linear) = {mse(base_avg, res):.2e}")
        
    print("Гаусс:")
    base_gauss = gauss_filter(img_gray, linear) 
    for name, func in traversals.items():
        res = gauss_filter(img_gray, func)
        print(f"MSE({name} vs linear) = {mse(base_gauss, res):.2e}")

Усреднение:
MSE(linear vs linear) = 0.00e+00
MSE(spiral vs linear) = 0.00e+00
MSE(zigzag vs linear) = 0.00e+00
Гаусс:
MSE(linear vs linear) = 0.00e+00
MSE(spiral vs linear) = 0.00e+00
MSE(zigzag vs linear) = 0.00e+00
